# Davinci Notebook 15 — Repository Audit & Integration Checks

Run after cloning the repository into `/content/Davinci` (or from the repository root). This notebook validates notebook JSON, compiles pure-Python cells while safely skipping IPython/shell magic, compiles canonical package modules, and can run the CPU test suite. Static checks do not replace a GPU training run.


In [ ]:
import json, ast, pathlib, sys
ROOT=pathlib.Path('/content/Davinci') if pathlib.Path('/content/Davinci').exists() else pathlib.Path.cwd()
NOTEBOOK_DIR=ROOT/'notebooks'
print('Repository root:',ROOT)


In [ ]:
def python_source_without_magics(source):
    lines=source.splitlines()
    if any(line.lstrip().startswith('%%') for line in lines):
        return None, 'cell magic'
    output=[]; in_shell_continuation=False; skipped=False
    for line in lines:
        stripped=line.lstrip()
        if in_shell_continuation:
            output.append('')
            if not line.rstrip().endswith('\\'): in_shell_continuation=False
            skipped=True
        elif stripped.startswith(('!', '%')):
            output.append('')
            in_shell_continuation=line.rstrip().endswith('\\')
            skipped=True
        else:
            output.append(line)
    return '\n'.join(output), ('line magic/shell' if skipped else None)

def audit_notebooks(folder):
    report=[]
    for path in sorted(folder.glob('*.ipynb')):
        try: nb=json.loads(path.read_text(encoding='utf-8'))
        except Exception as e: report.append({'file':path.name,'json_error':str(e)}); continue
        errors=[]; code_count=0; skipped_count=0
        for ci,cell in enumerate(nb.get('cells',[])):
            if cell.get('cell_type')!='code': continue
            code_count+=1; raw=cell.get('source',[]); src=''.join(raw) if isinstance(raw,list) else raw
            clean,reason=python_source_without_magics(src)
            if clean is None: skipped_count+=1; continue
            if reason: skipped_count+=1
            try: ast.parse(clean,filename=f'{path.name}:cell-{ci}')
            except SyntaxError as e: errors.append({'cell':ci,'line':e.lineno,'message':e.msg})
        report.append({'file':path.name,'code_cells':code_count,'magic_cells_skipped_or_normalized':skipped_count,'syntax_errors':errors})
    return report

report=audit_notebooks(NOTEBOOK_DIR)
print(json.dumps(report,indent=2))
bad=[x for x in report if x.get('syntax_errors') or x.get('json_error')]
print('Notebooks with static syntax/JSON issues:',len(bad))
if bad: raise RuntimeError('Notebook audit failed; inspect report above.')


## 2. Architecture integration checks
This checks for configuration consistency, not GPU memory feasibility or benchmark quality.

In [ ]:
def check_model_config(c):
 errors=[]
 if c.get('hidden_size')!=c.get('num_attention_heads',0)*c.get('head_dim',0): errors.append('hidden/head dimension mismatch')
 if c.get('num_attention_heads',0)%max(1,c.get('num_key_value_heads',0)): errors.append('GQA head divisibility mismatch')
 if c.get('head_dim',0)%2: errors.append('RoPE head dimension must be even')
 if c.get('num_hidden_layers',0)<=0 or c.get('vocab_size',0)<=0: errors.append('layer/vocab count invalid')
 return errors
print('Fill config and run checks; tokenizer vocabulary and model configuration must match exactly.')


## 3. Required execution tests
- Run Notebook 5 smoke tests and checkpoint round-trip.
- Run Notebook 9 cached decode parity on CPU and GPU.
- Run Notebook 10 optimizer smoke-test and checkpoint resume.
- Run Notebook 6 data pipeline on a small corpus with separate train/validation documents.
- Run Notebook 12 SFT mask test with actual tokenizer special tokens.
- Run Notebook 13 schema audit on a curated trajectory file.
- Run Notebook 14 sampling test and connect cached generation.
- Test the entire pipeline with a tiny model and dataset before allocating full target weights.


## 4. Canonical package and automated tests
This executes the short CPU regression suite. It does not run full training or validate 500K context.


In [ ]:
import subprocess, importlib.util
package_files=sorted((ROOT/'src'/'davinci').glob('*.py'))
for path in package_files:
    compile(path.read_text(encoding='utf-8'),str(path),'exec')
print('Compiled canonical package files:',[p.name for p in package_files])

if importlib.util.find_spec('pytest') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','pytest'],check=True)
test_paths=['tests/test_model.py','tests/test_train.py','tests/test_long_context.py','tests/test_hierarchical_retrieval.py','tests/test_tool_call.py','tests/test_generation.py','tests/test_notebook_integrity.py']
result=subprocess.run([sys.executable,'-m','pytest','-q',*test_paths],cwd=ROOT,text=True)
if result.returncode:
    raise RuntimeError(f'Pytest failed with exit code {result.returncode}')
print('CPU regression suite completed successfully.')


## 5. Required environment-level validation
- Run the pretraining notebook with the tiny preset first and a small, licensed dataset.
- Record GPU model, VRAM, PyTorch/CUDA versions, peak memory, tokens/sec, and validation loss.
- Verify tokenizer/model vocabulary agreement and dataset provenance before training.
- Run long-context and retrieval task evaluations before making any 500K-context claim.
- Do not treat a passing CPU test suite as evidence of GPU training feasibility or model quality.
